In [163]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../titanic_data_updated.csv")
df.sample(4)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
151,152,yes,first,"Pears, Mrs. Thomas (Edith Wearne)",female,22.0,1,0,113776,66.6000,C2,S
215,216,yes,first,"Newell, Miss. Madeleine",female,31.0,1,0,35273,113.2750,D36,C
766,767,no,first,"Brewe, Dr. Arthur Jackson",male,NaN,0,0,112379,39.6000,NaN,C
96,97,no,first,"Goldschmidt, Mr. George B",male,71.0,0,0,PC 17754,34.6542,A5,C


In [164]:
df.drop(['PassengerId','Name','Ticket'], axis=1,inplace=True) # dropping this columns as they do not have any impact on survival rate

In [165]:
df.sample(2)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
670,yes,second,female,40.0,1,1,39.0,NaN,S
178,no,second,male,30.0,0,0,13.0,NaN,S


### Seperate Features and Target columns

In [166]:
# Features columns
x = df.drop('Survived',axis=1)
x.sample(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
865,second,female,42.0,0,0,13.0,NaN,S
56,second,female,21.0,0,0,10.5,NaN,S


In [167]:
# Target column
y = df['Survived']
display(y)

0       no
1      yes
2      yes
3      yes
4       no
      ... 
886     no
887    yes
888     no
889    yes
890     no
Name: Survived, Length: 891, dtype: object

### Train Test Split

In [168]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

In [169]:
display(x_train.head(3))
display(y_train.head(3))

,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
331,first,male,45.5,0,0,28.500,C124,S
733,second,male,23.0,0,0,13.000,NaN,S
382,third,male,32.0,0,0,7.925,NaN,S


331    no
733    no
382    no
Name: Survived, dtype: object

In [170]:
display(x_test.head(3))
display(y_test.head(3))

,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
709,third,male,NaN,1,1,15.2458,NaN,C
439,second,male,31.0,0,0,10.5000,NaN,S
840,third,male,20.0,0,0,7.9250,NaN,S


709    yes
439     no
840     no
Name: Survived, dtype: object

### Numerical Missing Values imputation using Pandas

In [171]:
x_train.isnull().sum()

Pclass        0
Sex           0
Age         140
SibSp         0
Parch         0
Fare          0
Cabin       553
Embarked      2
dtype: int64

Imputation with 'mean' value of age

In [172]:
x_train['age_mean_imputer'] = x_train['Age'].fillna(df['Age'].mean())

In [173]:
x_train.isnull().sum() #? notice this line -> age_mean_imputer      0

Pclass                0
Sex                   0
Age                 140
SibSp                 0
Parch                 0
Fare                  0
Cabin               553
Embarked              2
age_mean_imputer      0
dtype: int64

In [174]:
x_test['age_mean_imputer'] = x_test['Age'].fillna(df['Age'].mean())

In [175]:
x_test.isnull().sum()
#? age_mean_imputer    0

Pclass                0
Sex                   0
Age                  37
SibSp                 0
Parch                 0
Fare                  0
Cabin               134
Embarked              0
age_mean_imputer      0
dtype: int64

Imputation with 'median' value of age

In [176]:
x_train['age_median_imputer'] = x_train['Age'].fillna(df['Age'].median())
x_test['age_median_imputer'] = x_test['Age'].fillna(df['Age'].median())

In [177]:
# x_train.isnull().sum()
x_test.isnull().sum()
#? age_median_imputer      0

Pclass                  0
Sex                     0
Age                    37
SibSp                   0
Parch                   0
Fare                    0
Cabin                 134
Embarked                0
age_mean_imputer        0
age_median_imputer      0
dtype: int64

**Imputation using in ML way- using Scikit-Learn**

In [178]:
from sklearn.impute import SimpleImputer

age_imputer = SimpleImputer(missing_values=np.nan,strategy='mean');
age_imputer.fit(x_train[['Age']])
x_train['Age'] = age_imputer.transform(x_train[['Age']])

In [179]:
x_test['Age'] = age_imputer.transform(x_test[['Age']])

In [180]:
x_train.isnull().sum()

Pclass                  0
Sex                     0
Age                     0
SibSp                   0
Parch                   0
Fare                    0
Cabin                 553
Embarked                2
age_mean_imputer        0
age_median_imputer      0
dtype: int64

In [181]:
x_test.isnull().sum()

Pclass                  0
Sex                     0
Age                     0
SibSp                   0
Parch                   0
Fare                    0
Cabin                 134
Embarked                0
age_mean_imputer        0
age_median_imputer      0
dtype: int64

In [182]:
x_train.drop(['age_mean_imputer','age_median_imputer'],axis=1,inplace=True)
x_test.drop(['age_mean_imputer','age_median_imputer'],axis=1,inplace=True)

In [183]:
# x_train.sample(2)
x_test.sample(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
457,first,female,29.498846,1,0,51.8625,D21,S
211,second,female,35.000000,0,0,21.0000,NaN,S


### Categorical missing values imputation

In [184]:
from sklearn.impute import SimpleImputer

embark_imputer = SimpleImputer(missing_values=np.nan,strategy='most_frequent');
embark_imputer.fit(x_train[['Embarked']])

x_train['Embarked'] = embark_imputer.transform(x_train[['Embarked']]).ravel();

# for test data
x_test['Embarked'] = embark_imputer.transform(x_test[['Embarked']]).ravel();

In [185]:
x_train.isnull().sum()
# x_test.isnull().sum()

Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Cabin       553
Embarked      0
dtype: int64

### Imputing missing value with 'missing' string for cabin column

In [186]:
cabin_imputer = SimpleImputer(missing_values=np.nan, strategy='constant',fill_value='Missing',add_indicator=True)
cabin_imputer.fit(x_train[['Cabin']])

x_train[['Cabin','Cabin_Missing_Indicator']] = cabin_imputer.transform(x_train[['Cabin']]);

x_test[['Cabin','Cabin_Missing_Indicator']] = cabin_imputer.transform(x_test[['Cabin']]);

In [187]:
x_train.isnull().sum()

Pclass                     0
Sex                        0
Age                        0
SibSp                      0
Parch                      0
Fare                       0
Cabin                      0
Embarked                   0
Cabin_Missing_Indicator    0
dtype: int64

In [188]:
x_train.sample(3)

,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,Cabin_Missing_Indicator
713,third,male,29.0,0,0,9.4833,Missing,S,True
175,third,male,18.0,1,1,7.8542,Missing,S,True
406,third,male,51.0,0,0,7.7500,Missing,S,True


In [189]:
x_train.to_csv("./Outlier Detection/x_train.csv",index=True);
y_train.to_csv("./Outlier Detection/y_train.csv",index=True);
x_test.to_csv("./Outlier Detection/x_test.csv",index=True);
y_test.to_csv("./Outlier Detection/y_test.csv",index=True);